In [ ]:
# !pip install pymssql
# !pip install voila

# When you install voila, restart Jupyter, then refresh page to get the necessary icon on the toolbar.
# I'm in a docker container, so I used this command in PowerShell: docker restart <container-name>

# To open the app, comment out the install lines, then click the new icon that appears in the toolbar above.

In [2]:
import pymssql
import pandas as pd
import ipywidgets as widgets
import re
import warnings
from IPython.display import display, HTML
from ipywidgets import Dropdown, HBox

warnings.filterwarnings('ignore')

# docker inspect <sql-server-container-name> | Select-String "IPAddress"
conn = pymssql.connect(
    server='172.18.0.2',
    port=1433,
    user='sa',
    password='SU2orange!',
    database='StellarFilms'
)

In [3]:
lastnames_df = pd.read_sql('select distinct person_lastname from talent_stats order by person_lastname', conn)
firstnames_df = pd.read_sql('select distinct person_firstname from talent_stats order by person_firstname', conn)
genres_df = pd.read_sql('select distinct genre_name from genres order by genre_name', conn)

In [ ]:
display(HTML("""
    <div style='background: linear-gradient(135deg, #1a0f3c, #3d2b7a); 
                padding: 20px; border-radius: 10px; margin-bottom: 20px;'>
        <h1 style='color: goldenrod; margin:0;'>Stellar Films Studios</h1>
        <br>
        <h2 style='color: white; margin:0;'>Greenlight Recommendations</h2>
        <p style='color: #ccc; margin:0;'>Based on Directors, Genres, & Locations</p>
    </div>
"""))

firstname_dd = Dropdown(
    options=[''] + firstnames_df.values.flatten().tolist(),
    value='',
    layout=widgets.Layout(width='16%'), 
)

lastname_dd = Dropdown(
    options=[''] + lastnames_df.values.flatten().tolist(),
    value='',
    layout=widgets.Layout(width='16%'), 
)

genre_dd = Dropdown(
    options=[''] + genres_df.values.flatten().tolist(),
    value='',
    layout=widgets.Layout(width='16%'), 
)

firstname_label = widgets.HTML("<b style='display:inline-block; width:80px;'>First Name:</b>")
lastname_label = widgets.HTML("<b style='display:inline-block; width:80px;'>Last Name:</b>")
genre_label = widgets.HTML("<b style='display:inline-block; width:80px;'>Genre:</b>")

firstname_row = HBox([firstname_label, firstname_dd])
lastname_row = HBox([lastname_label, lastname_dd])
genre_row = HBox([genre_label, genre_dd])

# button = widgets.Button(description="Submit", button_style='info', icon='search')
button = widgets.Button(
    description="Submit",
    icon='search',
    style=widgets.ButtonStyle(button_color='#3d2b7a', text_color='white')
)

output = widgets.Output()

def on_button_clicked(b):
    with output:
        output.clear_output()
        try:
            query = f"exec dbo.p_greenlight @director_firstname='{firstname_dd.value}', @director_lastname='{lastname_dd.value}', @genre='{genre_dd.value}'"
            df = pd.read_sql(query, conn)
            display(df.style.hide(axis='index').format(precision=2).set_properties(**{
                'background-color': '#f5f5f5',
                'border': '1px solid #ddd',
                'padding': '8px',
                'width': '30%'
            }).set_table_styles([{
                'selector': 'th',
                'props': [('background-color', '#000000'), ('color', 'white'), ('padding', '7px')]
            }]))
        except Exception as e:
            error_msg = str(e)
            match = re.search(r"b'(.+?)\DB-Lib", error_msg)
            if match:
                display(HTML(f"<p style='color:darkred; font-weight:bold;'>⚠️ {match.group(1)}</p>"))
            else:
                display(HTML(f"<p style='color:darkred; font-weight:bold;'>⚠️ {error_msg}</p>"))

button.on_click(on_button_clicked)

display(firstname_row, lastname_row, genre_row)
display(HTML("<br>"))
display(button)
display(HTML("<br>"))
display(output)